In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class GroupedQueryAttention(nn.Module):
    def __init__(self, embed_dim: int, num_q_heads: int, num_kv_heads: int):
        super().__init__()
        assert num_q_heads % num_kv_heads == 0, "Количество Query голов должно быть кратно KV головам"
        
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = embed_dim // num_q_heads
        self.num_groups = num_q_heads // num_kv_heads
        
        # Линейные слои. Заметь, что K и V имеют МЕНЬШЕ параметров на выходе!
        self.W_q = nn.Linear(embed_dim, num_q_heads * self.head_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, num_kv_heads * self.head_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, num_kv_heads * self.head_dim, bias=False)
        
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape:[Batch, Seq_Len, Embed_Dim]
        batch_size, seq_len, _ = x.shape
        
        # 1. Получаем Q, K, V
        # Q shape:[Batch, Seq_Len, Num_Q_Heads, Head_Dim]
        q = self.W_q(x).view(batch_size, seq_len, self.num_q_heads, self.head_dim)
        # K, V shape:[Batch, Seq_Len, Num_KV_Heads, Head_Dim]
        k = self.W_k(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim)
        v = self.W_v(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim)
        
        # 2. Повторяем K и V для соответствия количеству Query голов (Суть GQA!)
        # repeat_interleave копирует каждую KV-голову `num_groups` раз
        # k_expanded shape:[Batch, Seq_Len, Num_Q_Heads, Head_Dim]
        k_expanded = torch.repeat_interleave(k, repeats=self.num_groups, dim=2)
        v_expanded = torch.repeat_interleave(v, repeats=self.num_groups, dim=2)
        
        # Транспонируем для удобства умножения матриц: [Batch, Head, Seq_Len, Head_Dim]
        q = q.transpose(1, 2)
        k_expanded = k_expanded.transpose(1, 2)
        v_expanded = v_expanded.transpose(1, 2)
        
        # 3. Считаем стандартное скалярное произведение (Scaled Dot-Product Attention)
        # scores shape:[Batch, Num_Q_Heads, Seq_Len, Seq_Len]
        scores = torch.matmul(q, k_expanded.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Применяем причинную маску (Causal Mask) - опущено для краткости, берем просто softmax
        attn_weights = F.softmax(scores, dim=-1)
        
        # 4. Умножаем на Values
        # context shape:[Batch, Num_Q_Heads, Seq_Len, Head_Dim]
        context = torch.matmul(attn_weights, v_expanded)
        
        # Возвращаем исходную размерность: [Batch, Seq_Len, Embed_Dim]
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        
        return self.out_proj(context)

# Пример использования:
# Представим модель LLaMA-7B: 32 Query головы, но всего 8 KV голов (группы по 4)
embed_dim = 4096
gqa_layer = GroupedQueryAttention(embed_dim=embed_dim, num_q_heads=32, num_kv_heads=8)

# Тестовый тензор (Батч 2, 10 токенов)
dummy_x = torch.randn(2, 10, embed_dim)
output = gqa_layer(dummy_x)

print(f"Input shape: {dummy_x.shape}")
print(f"Output shape: {output.shape}") 
# KV-кэш для этой архитектуры будет весить ровно в 4 раза меньше, чем в классическом MHA!

Input shape: torch.Size([2, 10, 4096])
Output shape: torch.Size([2, 10, 4096])


: 